In [0]:
import os, random, sys, uuid
from datetime import datetime, timezone
sys.path.append(os.path.abspath('..'))

from faker import Faker
from pyspark.sql import functions as F
from src.config import CATALOGO
from src.raw_writer import gravar_eventos

dbutils.widgets.text('novos', '5')
dbutils.widgets.text('atualizacoes', '2')
novos = int(dbutils.widgets.get('novos'))
atualizacoes = int(dbutils.widgets.get('atualizacoes'))

fake = Faker('pt_BR')
PERFIS = ['CONSERVADOR', 'MODERADO', 'ARROJADO']
FAIXAS_RENDA = ['ATE_5K', 'DE_5K_A_15K', 'DE_15K_A_50K', 'ACIMA_50K']
agora = datetime.now(timezone.utc).isoformat()
eventos = []

for _ in range(novos):
    eventos.append({
        'id_evento': str(uuid.uuid4()),
        'tipo_evento': 'CADASTRO',
        'id_cliente': str(uuid.uuid4()),
        'nome': fake.name(),
        'cpf': fake.cpf(),
        'email': fake.email(),
        'data_nascimento': fake.date_of_birth(minimum_age=18, maximum_age=80).isoformat(),
        'cidade': fake.city(),
        'uf': fake.estado_sigla(),
        'perfil_investidor': random.choice(PERFIS),
        'faixa_renda': random.choice(FAIXAS_RENDA),
        'evento_em_utc': agora,
    })

tabela = f'{CATALOGO}.bronze.clientes'
if atualizacoes > 0 and spark.catalog.tableExists(tabela):
    sorteados = (spark.table(tabela)
                 .filter("tipo_evento = 'CADASTRO'")
                 .select('id_cliente').distinct()
                 .orderBy(F.rand()).limit(atualizacoes).collect())
    for linha in sorteados:
        eventos.append({
            'id_evento': str(uuid.uuid4()),
            'tipo_evento': 'ATUALIZACAO',
            'id_cliente': linha.id_cliente,
            'cidade': fake.city(),
            'uf': fake.estado_sigla(),
            'perfil_investidor': random.choice(PERFIS),
            'evento_em_utc': agora,
        })

arquivo = gravar_eventos('clientes', eventos)
print(f'{len(eventos)} eventos de clientes gravados em {arquivo}')

In [0]:
%sql
select * from read_files('/Volumes/corretora/raw/landing/sintetico/clientes/dt_coleta=2026-11-03/', format => "json")